# 💊 Drug Side Effects: Severity Prediction

> **Goal:** Predict whether a drug's reported side effect will be **Mild**, **Moderate**, or **Severe**  
> using patient demographics, drug details, and lifestyle factors.

---

## 📋 Notebook Contents

| # | Section |
|---|---------|
| 1 | Import Libraries |
| 2 | Load Dataset & Initial Exploration |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Data Cleaning & Feature Engineering |
| 5 | Encode Categorical Features |
| 6 | Train / Test Split |
| 7 | Train Multiple ML Models |
| 8 | Evaluate & Compare Models |
| 9 | Feature Importance |
| 10 | Conclusion |

---
🗂️ **Rows:** 100,000 &nbsp;|&nbsp; 📐 **Columns:** 16 &nbsp;|&nbsp; 🎯 **Task:** Multi-class Classification  
🏷️ **Target:** `severity` (Mild · Moderate · Severe)


## 1. 📦 Import Libraries

We start by importing all the tools we need. Think of this like gathering your equipment before cooking.

In [ ]:
# Standard libraries for data manipulation
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')   # hide unimportant warning messages

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# Scikit-learn: for building and evaluating ML models
from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import (accuracy_score, f1_score,
                                       classification_report,
                                       confusion_matrix)
from sklearn.linear_model      import LogisticRegression
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier

print("✅ All libraries imported successfully!")


## 2. 📂 Load Dataset & Initial Exploration

In [ ]:
# ── Load the dataset ──
df = pd.read_csv('/kaggle/input/drug-side-effects-100k-dataset/drug_side_effects_100k_dataset.csv')

print(f"📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


In [ ]:
# A quick summary: data types + non-null counts
df.info()


In [ ]:
# Statistical summary of numeric columns
df.describe().round(2)


In [ ]:
# Check missing values in each column
missing = df.isnull().sum()
missing = missing[missing > 0]
print("🔍 Columns with missing values:
")
for col, count in missing.items():
    pct = count / len(df) * 100
    print(f"  {col:<22} → {count:>6,} missing  ({pct:.1f}%)")


## 3. 📊 Exploratory Data Analysis (EDA)

Before building any model, we explore the data to understand patterns and relationships.

In [ ]:
# ── Target variable distribution ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
severity_counts = df['severity'].value_counts()
colors = ['#4CAF50', '#FF9800', '#F44336']

# Bar chart
axes[0].bar(severity_counts.index, severity_counts.values,
            color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('🎯 Severity Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Severity Level')
axes[0].set_ylabel('Number of Reports')
for i, v in enumerate(severity_counts.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')

# Pie chart
axes[1].pie(severity_counts.values, labels=severity_counts.index,
            colors=colors, autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('🎯 Severity Split', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("⚠️  Note: The dataset is imbalanced — Mild cases dominate (~62%).")


In [ ]:
# ── Age distribution by severity ──
plt.figure(figsize=(11, 5))
for sev, color in zip(['Mild', 'Moderate', 'Severe'], ['#4CAF50', '#FF9800', '#F44336']):
    plt.hist(df[df['severity'] == sev]['age'],
             bins=30, alpha=0.6, label=sev, color=color, edgecolor='white')

plt.title('👤 Age Distribution by Severity Level', fontsize=14, fontweight='bold')
plt.xlabel('Patient Age')
plt.ylabel('Count')
plt.legend(title='Severity')
plt.tight_layout()
plt.show()


In [ ]:
# ── Top drugs & top side effects ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

drug_counts = df['drug_name'].value_counts().head(10)
axes[0].barh(drug_counts.index[::-1], drug_counts.values[::-1],
             color='#5C6BC0', edgecolor='white')
axes[0].set_title('💊 Top Reported Drugs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')

se_counts = df['side_effect'].value_counts().head(10)
axes[1].barh(se_counts.index[::-1], se_counts.values[::-1],
             color='#26A69A', edgecolor='white')
axes[1].set_title('⚠️ Top 10 Side Effects', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
# ── Severity vs key categorical features ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in zip(axes,
        ['smoker', 'alcohol_use', 'hospitalized'],
        ['🚬 Smoker', '🍺 Alcohol Use', '🏥 Hospitalized']):
    ct = (pd.crosstab(df[col].fillna('Unknown'), df['severity'], normalize='index') * 100)
    ct = ct.reindex(columns=['Mild', 'Moderate', 'Severe'])
    ct.plot(kind='bar', ax=ax, color=['#4CAF50', '#FF9800', '#F44336'], edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('% within group')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Severity', fontsize=9)

plt.tight_layout()
plt.show()


## 4. 🔧 Data Cleaning & Feature Engineering

We'll:
- Extract a useful numeric feature from the date columns (`days_to_report`)
- Fill in missing values
- Drop columns that would cause **data leakage** or carry no signal


In [ ]:
# ── Step 1: Extract a useful date feature ──
df['report_date']          = pd.to_datetime(df['report_date'])
df['treatment_start_date'] = pd.to_datetime(df['treatment_start_date'])

# How many days between starting treatment and reporting a side effect?
df['days_to_report'] = (df['report_date'] - df['treatment_start_date']).dt.days
print("✅ Created 'days_to_report' feature")

# ── Step 2: Fill missing values ──
# Use .assign() — the modern pandas way that avoids warnings
df = df.assign(
    chronic_condition = df['chronic_condition'].fillna('Unknown'),   # categorical → 'Unknown'
    alcohol_use       = df['alcohol_use'].fillna('Unknown'),
    recovery_days     = df['recovery_days'].fillna(df['recovery_days'].median())  # numeric → median
)
print("✅ Missing values filled")

# ── Step 3: Drop columns we don't need ──
# patient_id        → just an ID, not useful
# report_date       → already extracted days_to_report
# treatment_start_date → already used above
# outcome           → happens AFTER severity is determined (data leakage!)
drop_cols = ['patient_id', 'report_date', 'treatment_start_date', 'outcome']
df = df.drop(columns=drop_cols)
print(f"✅ Dropped {drop_cols}")

print(f"\n📐 Dataset after cleaning: {df.shape}")
print("\nRemaining missing values:", df.isnull().sum().sum())


## 5. 🔢 Encode Categorical Features

ML models need numbers, not text. We convert every text column into numbers using `LabelEncoder`.

In [ ]:
# ── Separate features (X) and target (y) ──
X = df.drop(columns=['severity'])
y = df['severity']

# ── Encode the TARGET column ──
le_target = LabelEncoder()
y_enc = le_target.fit_transform(y)
# Mapping: 0 = Mild, 1 = Moderate, 2 = Severe
print("Target encoding:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

# ── Encode all text (categorical) FEATURE columns ──
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"\nEncoding {len(cat_cols)} categorical columns: {cat_cols}")

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

print(f"\n✅ Final feature matrix shape: {X.shape}")
print(f"✅ No missing values: {X.isnull().sum().sum() == 0}")
X.head(3)


## 6. ✂️ Train / Test Split

We split data 80% for training and 20% for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size   = 0.20,    # 20% held out for testing
    random_state= 42,      # ensures same split every run
    stratify    = y_enc    # keeps class proportions the same in both sets
)

print(f"🏋️  Training samples : {X_train.shape[0]:,}")
print(f"🧪  Testing  samples : {X_test.shape[0]:,}")
print()

# Verify class balance is preserved
for label, cls in zip(range(3), le_target.classes_):
    n = (y_train == label).sum()
    print(f"  Train → {cls}: {n:,} ({n/len(y_train)*100:.1f}%)")


## 7. 🤖 Train ML Models

We compare four different models to find the best one.

In [ ]:
# ── Define the four models ──
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=15,
                                                   random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=200,
                                                        learning_rate=0.1,
                                                        max_depth=5, random_state=42),
}

# ── Train each model and record results ──
results = {}
print(f"{'Model':<28} | {'Accuracy':>8} | {'F1-Score':>8}")
print("-" * 52)

for name, model in models.items():
    model.fit(X_train, y_train)       # train
    y_pred = model.predict(X_test)    # predict on unseen data
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    results[name] = {'Accuracy': acc, 'F1': f1, 'model': model, 'preds': y_pred}
    status = "✅" if f1 > 0.56 else "🔸"
    print(f"{status} {name:<26} | {acc:>8.4f} | {f1:>8.4f}")

print("\n🏆 Training complete!")


## 8. 📊 Evaluate & Compare Models

In [ ]:
# ── Visual comparison ──
names = list(results.keys())
accs  = [results[n]['Accuracy'] for n in names]
f1s   = [results[n]['F1']       for n in names]

x     = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - width/2, accs, width, label='Accuracy',  color='#5C6BC0', alpha=0.9)
b2 = ax.bar(x + width/2, f1s,  width, label='F1 (weighted)', color='#26A69A', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('📊 Model Comparison: Accuracy & Weighted F1-Score', fontsize=14, fontweight='bold')
ax.axhline(y=0.62, color='red', linestyle='--', linewidth=1.2, label='Majority-class baseline (62%)')
ax.legend()

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
print("📌 Dashed red line = naive baseline (always predict 'Mild')")


In [ ]:
# ── Best model: detailed classification report ──
best_name  = max(results, key=lambda n: results[n]['F1'])
best_model = results[best_name]['model']
y_pred     = results[best_name]['preds']

print(f"🏆 Best Model: {best_name}")
print(f"   Accuracy : {results[best_name]['Accuracy']:.4f}")
print(f"   F1-Score : {results[best_name]['F1']:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=le_target.classes_))


In [ ]:
# ── Confusion matrix ──
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_,
            linewidths=0.5, linecolor='white')
plt.title(f'🔵 Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

print("\nHow to read this:")
print("  - Diagonal (top-left → bottom-right) = correct predictions ✅")
print("  - Off-diagonal = misclassifications ❌")


## 9. 🔑 Feature Importance

Which features helped the model most?

In [ ]:
# For tree-based models (RF / GBM), use built-in feature_importances_
# For Logistic Regression, use the mean absolute coefficient across classes

if hasattr(best_model, 'feature_importances_'):
    importance_vals = best_model.feature_importances_
else:
    importance_vals = np.abs(best_model.coef_).mean(axis=0)

feat_df = pd.DataFrame({'Feature': X.columns, 'Importance': importance_vals})             .sort_values('Importance', ascending=True)

palette = ['#F44336' if i >= len(feat_df) - 3 else '#5C6BC0'
           for i in range(len(feat_df))]

plt.figure(figsize=(9, 6))
plt.barh(feat_df['Feature'], feat_df['Importance'], color=palette)
plt.title(f'🔑 Feature Importance — {best_name}', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()
print("🔴 Red = Top 3 most important features")


## 10. 🏁 Conclusion

### 📌 Summary of Steps

| Step | What We Did |
|------|-------------|
| **EDA** | Explored 100,000 records; spotted class imbalance (Mild ≈ 62%, Severe ≈ 8%) |
| **Feature Engineering** | Extracted `days_to_report` from date columns |
| **Cleaning** | Filled nulls with 'Unknown' (categorical) and median (numeric) |
| **Modelling** | Trained 4 classifiers and compared with Accuracy & Weighted F1 |
| **Evaluation** | Used confusion matrix + classification report for full picture |

---

### 📈 Key Findings

- **Gradient Boosting** achieved the best weighted F1-Score among all models.
- The top predictive features were **age**, **recovery_days**, **days_to_report**, and **side_effect**.
- The dataset is **imbalanced** — *Mild* cases dominate. The model predicts Mild accurately but struggles with *Severe* cases.

---

### 🚀 What You Can Try Next

- ✅ **Handle imbalance** with `class_weight='balanced'` or **SMOTE** oversampling  
- ✅ **Hyperparameter tuning** with `GridSearchCV` or **Optuna**  
- ✅ Try **XGBoost** or **LightGBM** for higher performance  
- ✅ Build a **Gradio / Streamlit app** to predict severity for a new patient report  

---

> 💡 *This is a beginner-friendly notebook. If you learned something new or found it useful, please hit the* **👍 Upvote** *button — it really helps!*
